In [4]:
import os
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rcParams

INPUT_NPZ_DIR = "/home/ubuntu/dataset/holistic_result_comp2_aug_final/train"
OUTPUT_MP4_DIR = Path("/home/ubuntu/landmark_visualizations_mp4")
OUTPUT_MP4_DIR.mkdir(parents=True, exist_ok=True)

sample_npz_file = "0_68.npz"
sample_npz_path = os.path.join(INPUT_NPZ_DIR, sample_npz_file)
data = np.load(sample_npz_path, allow_pickle=True)


visual_data = {
    "pose": data["pose"].astype(np.float32, copy=True),
    "left_hand": data["left_hand"].astype(np.float32, copy=True),
    "right_hand": data["right_hand"].astype(np.float32, copy=True),
}

num_frames = visual_data["pose"].shape[0] if "pose" in visual_data else 0
if num_frames == 0:
    raise ValueError("data에 pose frame이 없습니다. sample_npz_path를 확인하세요.")

print(f"Frames to visualize: {num_frames}")

body_part_keys_to_plot = {
    "Pose": "pose",
    "Left Hand Rotated": "left_hand",
    "Right Hand Rotated": "right_hand",
}

body_part_colors = {
    "Pose": "blue",
    "Left Hand Rotated": "green",
    "Right Hand Rotated": "red",
}

fixed_xlim = (0, 1)
fixed_ylim = (1.2, 0)


def get_frame_points(frame_idx):
    points_by_part = {}

    for name, key in body_part_keys_to_plot.items():
        arr_for_frame = visual_data[key][frame_idx]
        if key == "pose":
            arr_for_frame = arr_for_frame[:23]

        valid_points_mask = arr_for_frame[:, 3] > 0
        valid_points = arr_for_frame[valid_points_mask]
        points_by_part[name] = valid_points[:, :2] if valid_points.size > 0 else np.empty((0, 2))

    return points_by_part


fig, ax = plt.subplots(1, 1, figsize=(10, 8))
scatters = {}

for name, color in body_part_colors.items():
    scatters[name] = ax.scatter(
        [],
        [],
        s=20,
        alpha=0.8,
        edgecolors="w",
        linewidths=0.5,
        label=name,
        color=color,
    )

ax.set_xlim(*fixed_xlim)
ax.set_ylim(*fixed_ylim)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.legend(title="Body Part")
ax.grid(True, linestyle="--", alpha=0.6)
title = ax.set_title("")


def update(frame_idx):
    points_by_part = get_frame_points(frame_idx)

    for name, offsets in points_by_part.items():
        scatters[name].set_offsets(offsets)

    ax.set_xlim(*fixed_xlim)
    ax.set_ylim(*fixed_ylim)
    title.set_text(f"Body fixed, hands rotated | Frame {frame_idx} | {sample_npz_file}")
    return [title, *scatters.values()]


fps = float(data.get("fps", 30))
fps = max(1, min(fps, 30))
interval = 1000 / fps

ani = animation.FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=interval,
    blit=False,
    repeat=True,
)

output_mp4_path = OUTPUT_MP4_DIR / f"{Path(sample_npz_file).stem}.mp4"
ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path is None:
    try:
        import imageio_ffmpeg
    except ImportError as exc:
        raise RuntimeError("MP4 저장에는 ffmpeg가 필요합니다. `sudo apt-get install ffmpeg` 또는 `pip install imageio-ffmpeg` 후 다시 실행하세요.") from exc
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
rcParams["animation.ffmpeg_path"] = ffmpeg_path
writer = animation.FFMpegWriter(fps=fps, bitrate=1800)
ani.save(str(output_mp4_path), writer=writer, dpi=120)
plt.close(fig)
print(f"Saved MP4: {output_mp4_path}")


Frames to visualize: 57
Saved MP4: /home/ubuntu/landmark_visualizations_mp4/0_68.mp4


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_DIR = "/home/ubuntu/FairyTaleSL/model"
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

# from data_ksl2 import (
#     INPUT_NPZ_DIR,
#     to_30fps,
#     drop_dummy_frames,
#     crop_video_by_hand_detection,
#     interpolate_short_gaps,
#     remove_short_hand_appearances,
#     person_center_scale,
#     gaussian_noise,
# )

sns.set_theme(style="whitegrid")
